# Goal: Use DCGAN trained on Pokemon images to generate new images that look like Pokemon, then implement a general Diffusion model fed the same data and compare the results

## Dataset: [kaggle-pokemon-images-dataset](https://www.kaggle.com/datasets/kvpratama/pokemon-images-dataset)

# DCGAN Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import os
from glob import glob
from PIL import Image
import time

In [ ]:
tf.random.set_seed(42)

## Check GPU Availability

In [ ]:
# We will be using Tensorflow with Apple's Metal API
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Available: {gpus}")
else:
    print("No GPU available, training might be slow")

# Get the Data

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 200
NOISE_DIM = 100
NUM_EXAMPLES_TO_GENERATE = 16

In [ ]:
DATASET_PATH = "./pokemon_jpg/pokemon_jpg"

In [ ]:
import zipfile

# Path to the zip file
zip_path = "./pokemon_jpg.zip"
extract_path = "./pokemon_jpg"

# Extract if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Zip file extracted successfully.")
else:
    print("Dataset already extracted.")

In [ ]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img = np.array(img)
    img = (img - 127.5) / 127.5  # Normalize to [-1,1]
    return img

In [ ]:
# Load Dataset
image_paths = glob(os.path.join(DATASET_PATH, "*.jpg"))
all_images = np.array([load_image(p) for p in image_paths])

print(f"Number of images loaded: {len(all_images)}")

# Create Dataset

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(all_images)
train_dataset = train_dataset.shuffle(buffer_size=60000).batch(BATCH_SIZE)

# Define the Generator

In [ ]:
def make_generator_model():
    model = tf.keras.Sequential([
        layers.Dense(8*8*512, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((8, 8, 512)),

        layers.Conv2DTranspose(256, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(128, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(3, (5,5), strides=(2,2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

# Define the Discriminator 

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=[IMG_SIZE, IMG_SIZE, 3]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

# Create & Warmup Models

In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()

In [ ]:
# Warm up models
dummy_noise = tf.random.normal([1, NOISE_DIM])
_ = generator(dummy_noise)

dummy_image = tf.random.normal([1, IMG_SIZE, IMG_SIZE, 3])
_ = discriminator(dummy_image)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

## Define Loss Functions

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_labels = tf.ones_like(real_output) * 0.9  # Label smoothing
    real_loss = cross_entropy(real_labels, real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

## Apply loss function

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

# Image Generation

## Initial Model

In [ ]:
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)

    if not os.path.exists('generated_images'):
        os.makedirs('generated_images')

    fig = plt.figure(figsize=(4,4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow((predictions[i] * 127.5 + 127.5).numpy().astype(np.uint8))
        plt.axis('off')

    plt.savefig(f'generated_images/image_at_epoch_{epoch:04d}.png')
    plt.show()
    plt.close(fig)

In [ ]:
seed = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, NOISE_DIM])

def train(dataset, epochs):
    gen_losses = []
    disc_losses = []

    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        gen_losses.append(gen_loss.numpy())
        disc_losses.append(disc_loss.numpy())

        generate_and_save_images(generator, epoch + 1, seed)

        print('Epoch {} completed in {:.2f}s, Gen Loss: {:.4f}, Disc Loss: {:.4f}'.format(
            epoch+1, time.time()-start, gen_loss, disc_loss
        ))

    # Final plot
    plot_losses(gen_losses, disc_losses)

In [ ]:
def plot_losses(gen_losses, disc_losses):
    plt.figure(figsize=(10,5))
    plt.plot(gen_losses, label='Generator Loss')
    plt.plot(disc_losses, label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.title('Generator and Discriminator Losses')
    plt.show()

In [ ]:
train(train_dataset, EPOCHS)

Initial thought is the generator is not fooling the discriminator, perhaps giving it more time via adjusting epochs might let it learn more. Model collapse doesn't seem to be occuring so there is room to optimize. 

In [ ]:
EPOCHS = 500

In [ ]:
train(train_dataset, EPOCHS)

Model output is still not ideal after additional epochs. The generator is having trouble fooling the discriminator. The discriminator at the same time is continually winning. Model might be stuck. The images provided are a small set < 1000, the model might need additional epochs to run while we strengthen the generator and weaken the discriminator. 

## Improved DCGAN Training (Version 2)
Applying label smoothing, noise injection, slower discriminator learning, and more epochs.m

In [ ]:
# Same as before
def make_generator_model_v2():
    model = tf.keras.Sequential([
        layers.Dense(8*8*512, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((8, 8, 512)),

        layers.Conv2DTranspose(256, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(128, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(3, (5,5), strides=(2,2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

In [ ]:
# Same as before
def make_discriminator_model_v2():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=[IMG_SIZE, IMG_SIZE, 3]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
generator_v2 = make_generator_model_v2()
discriminator_v2 = make_discriminator_model_v2()

dummy_noise = tf.random.normal([1, NOISE_DIM])
_ = generator_v2(dummy_noise)

dummy_image = tf.random.normal([1, IMG_SIZE, IMG_SIZE, 3])
_ = discriminator_v2(dummy_image)

generator_optimizer_v2 = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer_v2 = tf.keras.optimizers.Adam(4e-5)  # slower learning rate

In [ ]:
@tf.function
def train_step_v2(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator_v2(noise, training=True)

        # Force images to float32 to match noise dtype
        images = tf.cast(images, tf.float32)  # Important: Cast real images to float32
        
        # Add small Gaussian noise to real images
        real_images_noisy = images + 0.05 * tf.random.normal(shape=images.shape)

        real_output = discriminator_v2(real_images_noisy, training=True)
        fake_output = discriminator_v2(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator_v2.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator_v2.trainable_variables)

    generator_optimizer_v2.apply_gradients(zip(gradients_of_generator, generator_v2.trainable_variables))
    discriminator_optimizer_v2.apply_gradients(zip(gradients_of_discriminator, discriminator_v2.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
import os

def train_v2(dataset, epochs):
    gen_losses = []
    disc_losses = []
    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step_v2(image_batch)

        gen_losses.append(gen_loss.numpy())
        disc_losses.append(disc_loss.numpy())

        if (epoch + 1) % 100 == 0:
            generator_v2.save(f'generator_v2_epoch{epoch+1}.keras')
            discriminator_v2.save(f'discriminator_v2_epoch{epoch+1}.keras')

        generate_and_save_images(generator_v2, epoch + 1, seed)

        print('Epoch {} completed in {:.2f}s, Gen Loss: {:.4f}, Disc Loss: {:.4f}'.format(
            epoch+1, time.time()-start, gen_loss, disc_loss
        ))

    plot_losses(gen_losses, disc_losses)

In [ ]:
EPOCHS_V2 = 2000

# Start training
train_v2(train_dataset, EPOCHS_V2)

Interrupted training at epoch 1173. Model experienced mid-training mode collapse somewhere in epoch 700-800. Generator essentially learned ways to trick discriminator in prior epochs and committed to them after 700 epoch, stopping exploration from occurring.  

## Early Stopping DCGAN Training (Version 3)

Attempting to avoid mid-training model collapse.

Modifications: 
* Add noise to real and fake images
* Lower the generator learning rate
* Saves images every 100 epochs
* Added early stopping

In [ ]:
def make_generator_model_v3():
    model = tf.keras.Sequential([
        layers.Dense(8*8*512, use_bias=False, input_shape=(NOISE_DIM,)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((8, 8, 512)),

        layers.Conv2DTranspose(256, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(128, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(3, (5,5), strides=(2,2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

def make_discriminator_model_v3():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=[IMG_SIZE, IMG_SIZE, 3]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5,5), strides=(2,2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
generator_v3 = make_generator_model_v3()
discriminator_v3 = make_discriminator_model_v3()

# Warmup
dummy_noise = tf.random.normal([1, NOISE_DIM])
_ = generator_v3(dummy_noise)

dummy_image = tf.random.normal([1, IMG_SIZE, IMG_SIZE, 3])
_ = discriminator_v3(dummy_image)

# Optimizers
generator_optimizer_v3 = tf.keras.optimizers.Adam(8e-5)  # Lower learning rate
discriminator_optimizer_v3 = tf.keras.optimizers.Adam(4e-5)

In [ ]:
@tf.function
def train_step_v3(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generator produces fake images
        generated_images = generator_v3(noise, training=True)

        # Cast images properly
        images = tf.cast(images, tf.float32)

        # Add noise to real images
        real_images_noisy = images + 0.05 * tf.random.normal(shape=images.shape)

        # Add noise to fake images
        generated_images_noisy = generated_images + 0.05 * tf.random.normal(shape=generated_images.shape)

        real_output = discriminator_v3(real_images_noisy, training=True)
        fake_output = discriminator_v3(generated_images_noisy, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator_v3.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator_v3.trainable_variables)

    generator_optimizer_v3.apply_gradients(zip(gradients_of_generator, generator_v3.trainable_variables))
    discriminator_optimizer_v3.apply_gradients(zip(gradients_of_discriminator, discriminator_v3.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
import os

def train_v3(dataset, epochs, save_interval=100, early_stop_patience=200):
    gen_losses = []
    disc_losses = []
    best_gen_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step_v3(image_batch)

        gen_losses.append(gen_loss.numpy())
        disc_losses.append(disc_loss.numpy())

        # Save model every save_interval epochs
        if (epoch + 1) % save_interval == 0:
            generator_v3.save(f'generator_v3_epoch{epoch+1}.keras')
            discriminator_v3.save(f'discriminator_v3_epoch{epoch+1}.keras')
            generate_and_save_images(generator_v3, epoch+1, seed)
            print(f"Saved models at epoch {epoch+1}")

        # Simple early stopping: if generator loss increases for many epochs, stop
        if gen_loss.numpy() < best_gen_loss:
            best_gen_loss = gen_loss.numpy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch+1} — no gen loss improvement for {early_stop_patience} epochs.")
            break

        print('Epoch {} completed in {:.2f}s, Gen Loss: {:.4f}, Disc Loss: {:.4f}'.format(
            epoch+1, time.time()-start, gen_loss, disc_loss
        ))

    plot_losses(gen_losses, disc_losses)

In [ ]:
EPOCHS_V3 = 2000
train_v3(train_dataset, epochs=EPOCHS_V3, save_interval=100, early_stop_patience=200)

Early stopping kicked in but visually the model at 200 epochs did not seem to be collapsed. We could manually continue training the model, or adjust the way stopping is handled for future computations.

## Rerun Model 1

Model 1 was made without saving. In order to compare results we need to implement saving. 

In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()

# Optimizers
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
EPOCHS_V1 = 500

def train_v1(dataset, epochs, save_interval=100):
    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        # Save generated images every save_interval epochs like v3
        if (epoch + 1) % save_interval == 0:
            generate_and_save_images(generator, epoch + 1, seed)
            print(f"Generated images at epoch {epoch + 1}")

        print('Epoch {} completed in {:.2f}s, Gen Loss: {:.4f}, Disc Loss: {:.4f}'.format(
            epoch+1, time.time()-start, gen_loss, disc_loss
        ))

    # Save the generator model after full training
    generator.save('generator_epoch500.keras')
    print("Generator v1 model saved at epoch 500.")

# Run model
train_v1(train_dataset, EPOCHS_V1)

# Model Comparison

In [ ]:
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

NOISE_DIM = 100
NUM_IMAGES = 5  

generator_paths = {
    "v1": "generator_epoch500.keras",       
    "v2": "generator_v2_epoch500.keras",     
    "v3": "generator_v3_epoch200.keras"      
}

# Generate fixed noise once
fixed_seed = tf.random.normal([NUM_IMAGES, NOISE_DIM])

# Load models and generate samples
generated_images = {}

for version, path in generator_paths.items():
    print(f"Loading Generator {version} from {path}...")
    model = load_model(path)
    predictions = model(fixed_seed, training=False)
    generated_images[version] = predictions

# Comparison Grid
fig, axes = plt.subplots(len(generator_paths), NUM_IMAGES, figsize=(NUM_IMAGES*3, len(generator_paths)*3))

for row_idx, (version, images) in enumerate(generated_images.items()):
    for col_idx in range(NUM_IMAGES):
        ax = axes[row_idx, col_idx]
        img = images[col_idx]
        img = (img * 127.5 + 127.5).numpy().astype(np.uint8)  # Rescale to [0, 255]
        ax.imshow(img)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(f'Version {version}', fontsize=14, rotation=0, labelpad=60, va='center')

plt.suptitle("Comparison of Generator Outputs (v1 vs v2 vs v3)", fontsize=16)
plt.tight_layout()
plt.show()

# GAN and DCGAN Results

Initially, Version 1 (v1) implemented a basic Deep Convolutional GAN (DCGAN) architecture without any stabilization techniques.
While initial training showed some progress, the model ultimately never learned well enough to produce a useable image. In face the images were rather plain. 

In Version 2 (v2), I introduced key improvements:
* Label smoothing for real samples
* Noise injection into real images
* Slower discriminator learning rate relative to the generator

These changes resulted in visibly more stable training dynamics.
Generator and discriminator losses remained closer together, and mode collapse was delayed compared to v1.
However, extended training still eventually led to loss of diversity in outputs, highlighting the need for additional training controls. Mode collapse was observed after approximately 700 epochs, where generated images degraded into colored blobs.
This outcome demonstrated the inherent instability of GAN training, particularly on relatively small datasets, and may reflect my changes on the generator and discriminator. 

Finally, in Version 3 (v3), I applied a more complete stabilization strategy:
* Noise injection to both real and fake images
* Slightly slower generator learning rate
* Early stopping based on generator loss stagnation
* Periodic model checkpointing every 100 epochs

This version successfully avoided severe mode collapse within the training window, but training automatically stopped at epoch 217 after generator loss failed to meaningfully improve for 200 consecutive epochs. This showed me that early stopping with GAN is not necessarily simple or easy - visually we see that the model had not experinced mid-training model collapse at this point. Generated images at this stage retained much more visual diversity and structure compared to earlier versions, indicating more effective adversarial learning. Because of the adjustments made at v2 and v3, the models are saved at each 100 epochs - we would reuse the v3 model and continue training it if desired.

Across all experiments, important observations included:
* GAN training is highly sensitive to training balance between generator and discriminator.
* Visual inspection remains essential alongside loss tracking; loss curves alone do not guarantee high-quality outputs.
* Early stopping, when combined with diversity monitoring and periodic checkpointing, can be an effective strategy for practical GAN training workflows, but can also cut training too soon


# Next Steps

## Model Considerations

In order to create more realistic images, I looked for alternative and more advanced models that might meet the criteria. In particular, the next step from DCGan would be something like StyleGAN - in this case we have access to StyleGAN 2 and 3 (at this point in 2025). In addition, we also could look at diffusion models. 

## StyleGAN Pros and Cons

Pros: 

* High-quality image synthesis: StyleGAN2 produces sharp, photorealistic outputs with rich texture and detail, especially when trained sufficiently.
* Latent space disentanglement: Allows controlled manipulation of generated images by interpolating or modifying latent vectors.
* Adaptive augmentation (ADA): Helps stabilize training on small datasets by dynamically tuning augmentation strength.
* Modular and extensible architecture: Easier to fine-tune or expand due to well-separated mapping and synthesis networks.
* Progressive growing removed: Simplifies training while preserving high fidelity at full resolution.
* Training stability: More robust than earlier GANs due to updated normalization and architectural refinements.
* Wide community adoption: Extensive community support and research built around StyleGAN variants.

Cons:

* Sensitive to hyperparameters: Needs careful tuning of learning rates, augmentations, and architecture for optimal results.
* Training time: Still computationally intensive, especially at 256×256 or higher resolution on CPU or non-CUDA backends.
* Resource usage: Memory-hungry due to high model complexity and resolution support.
* Limited support on Apple M1/M2+ GPUs: StyleGAN2 expects CUDA, requiring fallback or heavy patching on MPS.
* Latent space may produce artifacts: Especially during early stages of training or with underrepresented data diversity.

## Diffusion Pros and Cons

Pros:
    
* Excellent mode coverage: Captures a wider range of variations compared to GANs, reducing mode collapse.
* Superior text/image fidelity: SOTA models like Stable Diffusion rival GANs in quality and diversity, especially with text conditioning.
* Training stability: More stable due to likelihood-based training, no adversarial discriminator.
* Easier scaling: Can improve with more compute and data without requiring architectural rework.
* High compatibility with MPS and modern hardware: Works well on Apple Silicon with frameworks like Hugging Face diffusers or PyTorch MPS support.

Cons: 

* Slower generation: Requires many sampling steps (e.g., 25–100+), making inference time much longer than GANs.
* Harder to train from scratch: Training diffusion models from scratch is data- and compute-intensive.
* Limited latent control: Harder to manipulate specific attributes unless trained with guidance or classifiers.
* Model size and disk footprint: Larger model checkpoints and pipelines than many GAN variants.

## Model Decision

These models are being run with Apple Silicon, an ARM processor, using the Metal GPU backend. What this means for StlyeGAN is that heavy modification to the training files is required. In this case, after making sufficient changes to sections to force either Metal OR CPU in non-metal supported sections, as well as changing float64 type to float32 for ARM support, ultimately the grid-based portion of StyleGAN2 was not supported on Apple Silicon. 

As such, we move to the Diffusion-based models that are more apt to support Apple Silicon. I will take advantage of Stable Diffusion 1.5 with pretraining to avoid compiling a model from scratch for days-weeks.

# Stable Diffusion Model

[DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation](https://arxiv.org/abs/2208.12242)

[LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)



U-Net Architecture introduced in paper [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) (DDPM)



![U-Net architecture](https://nn.labml.ai/unet/unet.png)

I chose to implement Stable Diffusion 1.5 using the Hugging Face diffusers library, enhanced with DreamBooth and LoRA support for efficient fine-tuning. This approach combines the strengths of diffusion-based generation with specialized tools for personalized image synthesis.

Stable Diffusion is a latent diffusion model that generates images by denoising a latent representation over multiple steps. It is trained to reconstruct high-quality images from noisy inputs, effectively learning a powerful generative distribution over natural images. Version 1.5 is a widely adopted baseline due to its balance of speed, quality, and community support.

DreamBooth is a fine-tuning technique that allows the model to learn new concepts from a small number of images (typically 3–10). It is ideal for personalizing the model to generate images of specific characters or objects (e.g., a particular Pokémon), while retaining general generation capabilities.

Low-Rank Adaptation (LoRA) is a parameter-efficient training method. Instead of updating all weights during fine-tuning, LoRA adds low-rank matrices to key weight matrices, reducing memory and compute requirements. This makes it feasible to fine-tune large models like Stable Diffusion even on consumer-grade GPUs or Apple Silicon.

The Hugging Face diffusers library provides a modular and well-documented API to load, modify, and train diffusion models. It supports many diffusion algorithms and integrates well with tools like Accelerate, Transformers, and Model Hub.

MPS stands for Metal Performance Shaders, Apple’s GPU framework for accelerating deep learning workloads on macOS devices with Apple Silicon (M1/M2/M3). PyTorch and Hugging Face support MPS through a backend that allows models to be trained and inferred using the Apple GPU, which improves performance significantly over CPU-only training.

## Key Differences Between Stable Diffusion and DCGAN

Noise Injection

* Stable Diffusion: Applies noise iteratively using a Gaussian noise schedule over many time steps.
* DCGAN: Injects noise once at the start via a random latent vector.

Generation Path
* Stable Diffusion: Gradually denoises a noisy latent vector through multiple steps to produce an image.
* DCGAN: Directly maps the noise vector to an image in a single forward pass through the generator.

Output Resolution Control
* Stable Diffusion: Operates in a lower-dimensional latent space and can generate high-resolution outputs using a separate decoder.
* DCGAN: Output resolution is fixed by the architecture and difficult to upscale without artifacts.

Training Stability
* Stable Diffusion: More stable since it avoids adversarial training and uses a reconstruction loss.
* DCGAN: Training can be unstable due to adversarial loss dynamics (e.g., mode collapse, vanishing gradients).

Quality of Results
* Stable Diffusion: Produces sharp, high-fidelity images with diverse content and structure.
* DCGAN: Tends to generate blurrier images, especially at higher resolutions, and may lack diversity.

# Diffusion Imports

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
print(torch.backends.mps.is_available())  # Should return True

In [ ]:
!pip install diffusers transformers accelerate safetensors peft
!pip install datasets ftfy tensorboard

In [ ]:
# Clone the repo
!git clone https://github.com/huggingface/diffusers.git

# Install it in editable mode
!pip install -e ./diffusers

In [ ]:
# Default values
# Running this locally will let you select options
from accelerate.utils import write_basic_config

write_basic_config()

Answer the prompts:

* Compute environment: This machine
* Type of training: No distributed training
* Mixed precision: No (or “fp16” if running on Colab later)
* Which accelerator: MPS
* Use DeepSpeed: No

I now have: 

* MPS-accelerated PyTorch
* HuggingFace diffusers with DreamBooth + LoRA support
* Your Mac’s GPU will be used for training

# DreamBooth/Lora Formatting

Folder structure needs to be in this format for DreamBooth: 

pokemon_lora/

└── instance_images/

    ├── pikachu1.jpg
    ├── charizard2.jpg
    └── ...

In [ ]:
from PIL import Image
import os

INPUT_DIR = "pokemon_jpg/pokemon_jpg"
OUTPUT_DIR = "pokemon_lora/instance_images"
TARGET_SIZE = (512, 512)

os.makedirs(OUTPUT_DIR, exist_ok=True)

for filename in os.listdir(INPUT_DIR):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        img = Image.open(os.path.join(INPUT_DIR, filename)).convert("RGB")
        img = img.resize(TARGET_SIZE)
        img.save(os.path.join(OUTPUT_DIR, filename))

print("Resized images saved to:", OUTPUT_DIR)

In order to use huggingface diffusion model, we have to login to our account and provide a generated token

# Diffusion Model Creation

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")

In [ ]:
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="pokemon_lora/instance_images" \
  --output_dir="pokemon_lora_output" \
  --instance_prompt="a sks pokémon" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --checkpointing_steps=100 \
  --seed=42 \
  --mixed_precision="no" \
  --validation_prompt="a sks pokémon" \
  --validation_epochs=100 \
  --report_to="tensorboard"

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch
from safetensors.torch import load_file
from pathlib import Path
import matplotlib.pyplot as plt

# === CONFIGURATION ===
model_id = "runwayml/stable-diffusion-v1-5"
lora_weights = "pokemon_lora_output/pytorch_lora_weights.safetensors"  # final weights
prompt = "a sks pokémon"
num_images = 4
guidance_scale = 7.5

# === LOAD PIPELINE ===
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    safety_checker=None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("mps")  # or "cpu" if needed

# === LOAD LoRA WEIGHTS ===
state_dict = load_file(lora_weights)
compatible_keys = [k for k in state_dict.keys() if "lora" in k]

In [ ]:
# Apply LoRA weights manually
pipe.unet.load_state_dict({k: v for k, v in state_dict.items() if "unet" in k}, strict=False)

print(f"Loaded LoRA weights from: {lora_weights}")

## First Run

In [ ]:
# === GENERATE IMAGES ===
images = pipe(prompt, num_inference_steps=50, guidance_scale=guidance_scale, num_images_per_prompt=num_images).images

# === DISPLAY RESULTS ===
fig, axs = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
for i, img in enumerate(images):
    axs[i].imshow(img)
    axs[i].axis("off")
plt.tight_layout()
plt.show()

The results are giving us Pokemon-like images, shaped from the dataset we fed the model, however they are intermixed with other information from the generalist diffusion model.

## Second Run

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch
from safetensors.torch import load_file
import matplotlib.pyplot as plt

# === CONFIGURATION ===
model_id = "runwayml/stable-diffusion-v1-5"
lora_weights = "pokemon_lora_output/pytorch_lora_weights.safetensors"
prompt = "a 2D sks creature in Pokémon style, full-body, centered, clean background"
negative_prompt = "photo, realistic, laptop, weapon, gun, human, text, signature, watermark, blurry, deformed"
num_images = 4
guidance_scale = 7.5
num_inference_steps = 50

# === LOAD PIPELINE ===
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    safety_checker=None,  # optionally disable safety checker
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("mps")  # use 'cpu' if you get device-related errors

# === LOAD LoRA WEIGHTS ===
state_dict = load_file(lora_weights)
pipe.unet.load_state_dict({k: v for k, v in state_dict.items() if "lora" in k}, strict=False)

print(f"✅ LoRA weights loaded from {lora_weights}")

# === GENERATE IMAGES ===
images = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_images_per_prompt=num_images,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale
).images

# === DISPLAY RESULTS ===
fig, axs = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
for i, img in enumerate(images):
    axs[i].imshow(img)
    axs[i].axis("off")
plt.tight_layout()
plt.show()

The second run leverages a negative prompt in an attempt to reject the unwanted aspects seen in the first run. In addition, the initial prompt used has been better defined. These changes resulted in better image generation. 

## Third Run

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch
from safetensors.torch import load_file
import matplotlib.pyplot as plt

# === CONFIGURATION ===
model_id = "runwayml/stable-diffusion-v1-5"
lora_weights = "pokemon_lora_output/pytorch_lora_weights.safetensors"
prompt = "a 2D sks creature in Pokémon style, full-body, centered, forest background"
negative_prompt = "photo, realistic, laptop, weapon, gun, human, text, signature, watermark, blurry, deformed"
num_images = 4
guidance_scale = 7.5
num_inference_steps = 50

# === LOAD PIPELINE ===
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    safety_checker=None,  # optionally disable safety checker
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("mps")  # use 'cpu' if you get device-related errors

# === LOAD LoRA WEIGHTS ===
state_dict = load_file(lora_weights)
pipe.unet.load_state_dict({k: v for k, v in state_dict.items() if "lora" in k}, strict=False)

print(f"✅ LoRA weights loaded from {lora_weights}")

# === GENERATE IMAGES ===
images = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_images_per_prompt=num_images,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale
).images

# === DISPLAY RESULTS ===
fig, axs = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
for i, img in enumerate(images):
    axs[i].imshow(img)
    axs[i].axis("off")
plt.tight_layout()
plt.show()

Finally in the third run we slightly change the prompt to provide a more consistent background for the images. The results are more consistent and convincing. The artistic style is more aligned and cohesive. 

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from safetensors.torch import load_file
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# === CONFIGURATION ===
model_id = "runwayml/stable-diffusion-v1-5"
checkpoint_steps = [300, 600, 900, 1000]  # Adjust as needed
images_per_checkpoint = 4
prompt = "a 2D sks creature in Pokémon style, full-body, centered, forest background"
negative_prompt = "photo, realistic, laptop, weapon, gun, human, text, signature, watermark, blurry, deformed"
guidance_scale = 7.5
num_inference_steps = 50

# === LOAD BASE PIPELINE ===
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    safety_checker=None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("mps")  # or "cpu" if needed

# === Generate and Display Grid of Images ===
n_rows = len(checkpoint_steps)
n_cols = images_per_checkpoint
fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))

for row, step in enumerate(checkpoint_steps):
    weights_path = f"pokemon_lora_output/checkpoint-{step}/pytorch_lora_weights.safetensors"
    if not Path(weights_path).exists():
        print(f"Checkpoint not found: {weights_path}")
        continue

    # Load LoRA weights
    state_dict = load_file(weights_path)
    pipe.unet.load_state_dict({k: v for k, v in state_dict.items() if "lora" in k}, strict=False)

    # Generate multiple images
    images = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_images_per_prompt=images_per_checkpoint,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale
    ).images

    # Plot them
    for col in range(images_per_checkpoint):
        ax = axs[row, col] if n_rows > 1 else axs[col]
        ax.imshow(images[col])
        ax.axis("off")
        if col == 0:
            ax.set_title(f"Step {step}", fontsize=14)

plt.tight_layout()
plt.show()

# Diffusion Model Results

The diffusion model used in this project was based on Stable Diffusion v1.5, implemented via Hugging Face’s diffusers library. We further customized the training using DreamBooth and LoRA (Low-Rank Adaptation), which allowed us to fine-tune the model on a relatively small dataset of Pokémon images while retaining base model generalization.

Highlights
* Model Base: Stable Diffusion v1.5 (pretrained text-to-image model)
* Fine-Tuning Method: DreamBooth + LoRA
* Training Platform: Apple M1/M2 GPU via MPS backend
* Image Dataset: ~800–1000 Pokémon images, cleaned and resized to 512x512

Observed Results
* The model successfully learned to generate Pokémon-themed images, especially when prompted with the specific fine-tuned token.
* Visual quality improved across training checkpoints, with clearer features and more consistent Pokémon-like structure appearing after a few hundred steps.
* Many outputs heavily favored Pikachu-like features, likely due to class imbalance or dataset bias.
* Some generations began to show object fusion artifacts, where Pokémon were blended with unrelated shapes or items—this was more common early in training or with ambiguous prompts.

Strengths
* High image fidelity: Images were sharp and coherent, even at low inference steps.
* Customizable outputs: Text prompts with trigger tokens reliably guided the output style.
* Training efficiency: LoRA reduced VRAM usage and allowed training on a Mac with MPS acceleration.
* Repeatability: The pretrained foundation ensured that results were not purely random even before fine-tuning.

Limitations
* Bias toward popular Pokémon (e.g., Pikachu) due to uneven representation in the dataset.
* MPS backend support is limited, resulting in occasional fallback to CPU and slower training performance.
* Some results still showed diffuse structure or lacked strong anatomical consistency without heavy prompt engineering.

# Conclusion 

Overall, the fine-tuned Stable Diffusion model provided strong, flexible image generation results. While artifacts were present early in training or in underrepresented categories, the outputs improved significantly with more steps and focused prompt usage. Compared to the StyleGAN2 model, the diffusion model offered greater prompt control and visual variety, albeit with longer training and generation times on CPU/MPS.

Overall, the fine-tuned Stable Diffusion model provided strong, flexible image generation results. While artifacts were present early in training or in underrepresented categories, the outputs improved significantly with more steps and focused prompt usage. Compared to the StyleGAN2 model, the diffusion model offered greater prompt control and visual variety, albeit with longer training and generation times on CPU/MPS.

Across this project, we explored multiple generative models to synthesize Pokémon-style images: a basic GAN, a DCGAN, and ultimately a fine-tuned Stable Diffusion model with DreamBooth and LoRA support.
* The GAN served as a foundational baseline but struggled with stability and resolution. It often failed to capture coherent shapes, showing mode collapse and blurry results during training.
* The DCGAN introduced convolutional layers, which noticeably improved structure and quality over the vanilla GAN. However, it was still limited in resolution (typically 64×64 or 128×128) and lacked fine detail in the generated Pokémon.
* In contrast, the Stable Diffusion model produced much more detailed and coherent images. It was able to synthesize realistic Pokémon-inspired creatures at high resolution (512×512), with flexible prompt-driven generation capabilities.

While GAN-based models were faster to train and easier to understand conceptually, they lacked the expressive power and controllability seen in the diffusion approach. On the other hand, the diffusion model—with DreamBooth and LoRA—allowed us to specialize a general-purpose generator to focus on a specific concept (Pokémon) without retraining from scratch.

Ultimately, the diffusion model proved to be the most effective solution for high-quality, personalized image generation—making it the clear winner in terms of both visual fidelity and adaptability.